# 12 — Specialty Charts
**Goal:** Build the chart types that show up in business and growth contexts
but are not in stock matplotlib: waterfall, sankey, treemap, funnel, bump,
and a calendar heatmap.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from matplotlib.patches import Rectangle

np.random.seed(66)

## 1. Waterfall — the decomposition chart

A waterfall decomposes a starting value into a series of positive and negative
contributions ending at a final value. The classic case: *revenue bridge from
Q1 to Q2*.

In [ ]:
def waterfall(ax, labels, values, total_labels=('Start', 'End')):
    running = np.cumsum([0] + values[:-1])
    arr_v = np.array(values)
    bottoms = np.where(arr_v >= 0, running, running + arr_v)
    colors = ['gray' if l in total_labels else
              ('seagreen' if v >= 0 else 'crimson') for l, v in zip(labels, values)]
    for i, (l, v) in enumerate(zip(labels, values)):
        ax.bar(i, abs(v), bottom=bottoms[i], color=colors[i],
               edgecolor='white', width=0.7)
        offset = abs(max(values)) * 0.03
        ax.text(i, bottoms[i] + abs(v) + (offset if v >= 0 else -offset*2),
                f'{v:+}', ha='center', fontsize=9,
                va='bottom' if v >= 0 else 'top')
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=0)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', color='lightgray', alpha=0.5)

labels = ['Q1', 'New', 'Churn', 'Expansion', 'Price', 'Q2']
values = [120,   18,    -8,        12,          4,     146]
fig, ax = plt.subplots(figsize=(9, 4.5))
waterfall(ax, labels, values, total_labels=('Q1', 'Q2'))
ax.set_title('Revenue bridge Q1 → Q2', loc='left', weight='bold')
plt.tight_layout(); plt.show()

## 2. Treemap — hierarchical part-to-whole

A treemap tiles a rectangle with sub-rectangles whose area encodes value. The
most useful Python implementation is `plotly.express.treemap`.

In [ ]:
df = pd.DataFrame({
    'region' : ['EMEA','EMEA','EMEA','APAC','APAC','AMER','AMER','AMER','AMER'],
    'channel': ['Search','Social','Display','Search','Social','Search','Social','Display','Email'],
    'rev'    : [42, 18, 12, 25, 14, 38, 22, 19, 11],
})
fig = px.treemap(df, path=['region','channel'], values='rev',
                 color='rev', color_continuous_scale='YlGnBu')
fig.update_layout(title='Revenue by region × channel', width=700, height=400)
fig.show()

## 3. Sunburst — the radial treemap

Same data as a treemap, but layered radially. Best for **navigating** a
hierarchy, not measuring exact values.

In [ ]:
fig = px.sunburst(df, path=['region','channel'], values='rev',
                  color='rev', color_continuous_scale='YlGnBu')
fig.update_layout(title='Sunburst — same data, radial', width=600, height=500)
fig.show()

## 4. Sankey — flows between states

A Sankey shows how a quantity flows between categorical states, with the width
of each band proportional to the amount. Use for: user journeys,
attribution, energy/material flow.

In [ ]:
fig = go.Figure(go.Sankey(
    arrangement='snap',
    node=dict(
        pad=15, thickness=20,
        label=['Visit','Sign-up','Activate','Subscribe','Churn','Browse'],
        color='steelblue',
    ),
    link=dict(
        source=[0, 0, 1, 1, 2, 2, 3],
        target=[1, 5, 2, 5, 3, 5, 4],
        value =[100, 30,  60, 10, 50, 10, 5],
    ),
))
fig.update_layout(title='Sankey — user journey', width=750, height=350)
fig.show()

## 5. Funnel — the conversion chart

A funnel is a single-sequence sankey. Plotly Express has it built in.

In [ ]:
stages = pd.DataFrame({
    'stage' : ['Visit', 'Sign-up', 'Activate', 'Subscribe', 'Retain'],
    'value' : [10000, 4200, 2800, 1400, 1100],
})
fig = px.funnel(stages, x='value', y='stage', title='Conversion funnel')
fig.update_layout(width=650, height=380, yaxis=dict(autorange='reversed'))
fig.show()

## 6. Bump chart — ranking over time

A bump chart shows *ranking*, not value. Y is rank (1 = best). The chart
reveals *who overtook whom*.

In [ ]:
months = pd.date_range('2024-01-01', periods=8, freq='MS')
np.random.seed(7)
raw = np.cumsum(np.random.normal(0, 0.5, (4, 8)), axis=1) + np.arange(4).reshape(-1,1)*0.5
df = pd.DataFrame(raw, index=['Search','Social','Display','Email'], columns=months)
df = df.div(df.sum(axis=0), axis=1)
ranks = df.rank(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4.5))
colors = ['steelblue','crimson','seagreen','orange']
for (name, row), c in zip(ranks.iterrows(), colors):
    ax.plot(row.index, row.values, marker='o', lw=2, color=c, label=name)
    ax.text(row.index[-1], row.values[-1], f' {name}', color=c, va='center', fontsize=9)
ax.invert_yaxis()
ax.set_yticks(range(1, 5))
ax.set_title('Channel share rank over time (1 = leader)', loc='left', weight='bold')
ax.spines[['top','right']].set_visible(False)
ax.legend(frameon=False, loc='lower right')
plt.tight_layout(); plt.show()

## 7. Range / ribbon — uncertainty visualization

When the data is a *forecast* or *confidence interval*, plot the band, not just
the line. People ignore uncertainty if it is not drawn.

In [ ]:
t = np.arange(0, 50)
y = 100 + 2*t + np.random.normal(0, 5, 50)
sigma = 5 + t*0.4

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t, y, color='steelblue', lw=2, label='point forecast')
ax.fill_between(t, y - 1.96*sigma, y + 1.96*sigma,
                color='steelblue', alpha=0.2, label='95% interval')
ax.spines[['top','right']].set_visible(False)
ax.legend(frameon=False, loc='upper left')
ax.set_title('Forecast with 95% interval — show the uncertainty', loc='left', weight='bold')
plt.tight_layout(); plt.show()

## 8. Calendar heatmap — daily activity over a year

Useful for engagement and ops-style dashboards. Each row is a week, each cell a
day, color is value.

In [ ]:
dates = pd.date_range('2024-01-01', '2024-12-31', freq='D')
values = np.random.gamma(2, 5, len(dates))
cal = pd.DataFrame({'date': dates, 'value': values})
cal['week'] = ((cal['date'] - cal['date'].min()).dt.days // 7).astype(int)
cal['dow']  = cal['date'].dt.dayofweek
cal['week_of_year'] = cal['date'].dt.isocalendar().week.astype(int)

fig, ax = plt.subplots(figsize=(13, 4))
n_weeks = cal['week'].max() + 1
grid = np.full((7, n_weeks), np.nan)
for _, r in cal.iterrows():
    grid[r['dow'], r['week']] = r['value']
im = ax.imshow(grid, aspect='auto', cmap='YlGnBu')
ax.set_yticks(range(7))
ax.set_yticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'])
ax.set_xticks(range(0, n_weeks, 4))
ax.set_xticklabels([f'W{w}' for w in range(0, n_weeks, 4)], rotation=0, fontsize=8)
ax.set_title('Calendar heatmap — daily activity in 2024', loc='left', weight='bold')
fig.colorbar(im, ax=ax, label='value')
plt.tight_layout(); plt.show()

## 9. Streamgraph — flowing composition over time

Like a stacked area, but centered on a baseline. Visually striking, hard to
read for exact values. Use for *vibe* charts (composition shifting over time).

In [ ]:
t = np.linspace(0, 12, 200)
series = {f'cat {i}': np.sin(t + i) + i*0.3 + np.random.normal(0, 0.2, 200)
          for i in range(5)}
stack = np.vstack(list(series.values()))
stack = np.clip(stack, 0, None)
baseline = -stack.sum(axis=0) / 2
fig, ax = plt.subplots(figsize=(10, 4))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(series)))
for i, (name, _) in enumerate(series.items()):
    ax.fill_between(t, baseline, baseline + stack[i], color=colors[i], label=name, alpha=0.85)
    baseline = baseline + stack[i]
ax.set_yticks([]); ax.spines[['top','right','left']].set_visible(False)
ax.legend(frameon=False, ncol=5, loc='upper center', bbox_to_anchor=(0.5, -0.05))
ax.set_title('Streamgraph — flowing composition', loc='left', weight='bold')
plt.tight_layout(); plt.show()

## Summary

| Chart | Use |
|---|---|
| Waterfall | Bridge from start to end through contributions |
| Treemap | Hierarchical part-to-whole, area encoding |
| Sunburst | Hierarchy as radial tree (navigate, not measure) |
| Sankey | Flows between categorical states |
| Funnel | Single-sequence conversion |
| Bump | Rank change over time |
| Range / ribbon | Forecast uncertainty |
| Calendar | Daily / weekly / monthly patterns |
| Streamgraph | Aesthetic composition over time |

**Next:** `13_dash_dashboards.ipynb` — turn these charts into a web product.